In [1]:

import argparse
import logging
from types import SimpleNamespace
import os
from pathlib import Path
import csv
import sys
import math
import numpy as np
import pandas as pd
import time
import joblib
import json 
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
# sys.path.append('/Users/yiqin/Documents/PythonProjects/GraspDataProcessing/src')
# sys.path.append('D:\\PythonPrograms\\GraspDataProcessing\\src')
sys.path.append('D:\\PythonProjects\\GraspDataProcessing\\src')



In [9]:
%load_ext autoreload
%autoreload 2
import graspdataprocessing as gdp

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
config_file = 'config.toml'  # 配置文件的路径
config = gdp.load_config(config_file)

配置验证通过: cutoff_value=1e-09, initial_ratio=0.09


In [4]:
"""主程序逻辑"""
config.file_name = f'{config.conf}_{config.cal_loop_num}'
logger = gdp.setup_logging(config)
logger.info("机器学习训练程序启动")
execution_time = time.time()

gdp.setup_directories(config)
# 初始化结果文件
gdp.initialize_results_file(config, logger)

# 验证初始文件
gdp.validate_initial_files(config, logger)

logger.info(f"初始比例: {config.initial_ratio}")
logger.info(f"光谱项: {config.spetral_term}")

2025-06-10 17:17:23,348 - INFO - 机器学习训练程序启动
2025-06-10 17:17:23,348 - INFO - 成功加载初始CSFs文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4.c
2025-06-10 17:17:23,350 - INFO - 初始比例: 0.09
2025-06-10 17:17:23,350 - INFO - 光谱项: ['5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_9D', '5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_7D']


In [5]:
try:
    # 加载数据文件
    energy_level_data_pd, rmix_file_data, target_pool_csfs_data, raw_csfs_descriptors, cal_csfs_data, caled_csfs_indices_dict = gdp.load_data_files(config, logger)
    
    # 检查组态耦合
    cal_result = gdp.check_configuration_coupling(config, energy_level_data_pd, logger)
    logger.info("************************************************")

except Exception as e:
        logger.error(f"程序执行过程中发生错误: {str(e)}")
        raise

2025-06-10 17:17:24,799 - INFO - 加载能级数据: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level


Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level loaded.
Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level loaded.
file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level loaded
data file type: level data
energy levels data has been written into LEVEL/cv4odd1as3_odd4_1.level_level.csv csv file
Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.m loaded.
data file type: rmcdhf mix_coefficient_data
g92mix: G92MIX
 nblock = 1,       ncftot =   385600,          nw =  41,            nelec =   64


100%|██████████| 1/1 [00:00<00:00, 499.74it/s]
2025-06-10 17:17:24,807 - INFO - 加载 *.m 文件数据: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.m


cycle jblock = 1
 Block no. = 1, 2J+1 = 9, Parity = -1, No. of eigenvalues = 2, No. of CSFs = 385600

    Energy levels for ...
Rydberg constant is  109737.31568508

---------------------------------------------
 No Pos  J  Parity   Energy Total    Levels
                      (a.u.)         (cm^-1)
---------------------------------------------

  1  1   4   +    -11274.7413433        0.00
  2  0   4   +    -11274.7204780     4579.40


2025-06-10 17:17:30,984 - INFO - 加载初始 CSFs 文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4
2025-06-10 17:17:31,271 - INFO - 加载初始 CSFs 描述符文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4


Descriptors loaded from: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_descriptors.npy
Labels loaded from: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_descriptors_multi_block.npy
Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.c loaded.


2025-06-10 17:17:32,484 - INFO - 加载本轮计算 CSFs 文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.c
2025-06-10 17:17:32,495 - INFO - 加载本轮选择的 CSFs 的索引文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1_chosen_indices.pkl
2025-06-10 17:17:32,497 - INFO - cal_loop 1 组态耦合正确
2025-06-10 17:17:32,498 - INFO - ************************************************


In [6]:
caled_csfs_descriptors = gdp.generate_chosen_csfs_descriptors(config, caled_csfs_indices_dict, raw_csfs_descriptors, rmix_file_data, logger)

2025-06-10 17:17:35,416 - INFO - 保存本轮选择的 CSFs 的描述符文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1/cv4odd1as3_odd4_1.npy


Descriptors saved to: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1_descriptors.npy


In [7]:
stay_csfs_descriptors = gdp.get_stay_descriptors(raw_csfs_descriptors, caled_csfs_indices_dict)

In [12]:
X_stay = stay_csfs_descriptors.copy()

In [9]:
rmix_file_data.mix_coefficient_List[0]

array([[ 3.88368822e-01, -3.75787854e-01,  2.86599207e-01, ...,
         1.36427027e-07,  6.39849076e-07,  5.39727752e-08],
       [ 1.24814669e-02, -1.21756193e-02,  9.17219688e-03, ...,
        -6.31284283e-07,  5.70214263e-07,  3.64236044e-07]])

In [11]:
model, X_train, X_test, y_train, y_test, training_time, weight = gdp.train_model(config, caled_csfs_descriptors, rmix_file_data, logger)

2025-06-10 17:21:59,362 - INFO -              训练模型
2025-06-10 17:22:14,690 - INFO - Epoch [50/150], Loss: 0.024302
2025-06-10 17:22:29,592 - INFO - Epoch [100/150], Loss: 0.023029
2025-06-10 17:22:45,292 - INFO - Epoch [150/150], Loss: 0.022453
2025-06-10 17:22:45,293 - INFO -              预测与评估


(385600,)


2025-06-10 17:22:46,959 - INFO - 测试集预测结果:
2025-06-10 17:22:46,959 - INFO - AUC:0.8772400800733863, pr_auc:0.7791237672513263, f1:0.6775471018840754, accuracy:0.9651581950207468, precision:0.9701030927835051, recall:0.5205605753273096
2025-06-10 17:22:47,082 - INFO - 训练集预测结果:
2025-06-10 17:22:47,083 - INFO - AUC:0.9169697542746319, f1:0.6890254713078423, accuracy:0.965884336099585, precision:0.9877160284649271, recall:0.5290407477992558
2025-06-10 17:22:47,086 - INFO - 权重: [1, 10]
2025-06-10 17:25:19,987 - INFO - Epoch [50/150], Loss: 0.025652
2025-06-10 17:27:38,860 - INFO - Epoch [100/150], Loss: 0.025154
2025-06-10 17:29:52,223 - INFO - Epoch [150/150], Loss: 0.024926


In [19]:
X_test

array([[1., 1., 1., ..., 0., 0., 8.],
       [2., 0., 0., ..., 0., 0., 8.],
       [1., 1., 1., ..., 0., 0., 8.],
       ...,
       [2., 0., 0., ..., 0., 0., 8.],
       [2., 0., 0., ..., 0., 0., 8.],
       [1., 1., 1., ..., 0., 0., 8.]], dtype=float32)

In [14]:
evaluation_results = gdp.evaluate_model(
            model, X_train, X_test, y_train, y_test, X_stay, config, logger
        )

2025-06-10 17:32:29,736 - INFO - 开始预测与评估


PicklingError: Can't pickle <class 'graspdataprocessing.ANN.ANNClassifier'>: it's not the same object as graspdataprocessing.ANN.ANNClassifier